In [8]:
import pandas as pd
import glob
from pathlib import Path

import numpy as np
import random
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
from imblearn.under_sampling import RandomUnderSampler
import os
# IMPORTACIONES NUEVAS PARA LA PROYECCIÓN
import rasterio
from rasterio.transform import from_origin
import pyproj


Una vez descargados los archivos y puestos en las carpetas de su clase, se junta todo en un df que contiene la ubicación geografica y la etiqueta asignada a cada negocio:

    Ejemplo, si el archivo se puso en la carpeta de bajos ingresos, el modelo aceptara que ese negocio se asocia a bajos ingresos

In [9]:
base_dir = f"{Path(os.getcwd())}\DATA"
dfs = []
i = 1

for clase in [r'\C3_IngresosBajos',r'\C2_IngresosMedios',r'\C1_IngresosAltos']:
    # Construir ruta relativa
    ruta = base_dir + clase
    archivos_csv = glob.glob(str(ruta+"\*.csv"))
    print(archivos_csv)
    # Leer todos los CSV de esa carpeta
    df_temp = pd.concat([pd.read_csv(f, encoding='latin-1') for f in archivos_csv], ignore_index=True)
    df_temp["clase"] = i   # asigna número
    i += 1
    dfs.append(df_temp)

# Concatenar todo
df_final = pd.concat(dfs, ignore_index=True)
df_final = df_final[['Longitud','Latitud','clase']]
df_final.dropna(inplace=True)

# Guardar resultado en la misma carpeta del script
df_final.to_csv(base_dir + r"\DatosEconomicos\Coordenadas_Negocios_Con_Clase_Asociada.csv", index=False)

['c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\3b.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\3bTiendas.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\Agronomia.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\BodegAurrera.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\Elektra.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\FarmaciasSimilares.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C3_IngresosBajos\\Tortilleria.csv']
['c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C2_IngresosMedios\\Cubo_Granular_CDMX.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C2_IngresosMedios\\OXXO.csv']


C:\Users\xboxn\AppData\Local\Temp\ipykernel_31460\2339451709.py:11: DtypeWarning: Columns (6,7,8,9,11,12,13,14,15,16,17,18,19,20,22,23,24,25,26,27,28,30,31,32,33,34,35,36,37,38,40,41,42,43,44,45,46,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,71,72,73,75,76,77,78,79,80,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,98,99,100,101,103,105,106,107,108,109,110,112,113,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,131,132,133,134,135,136,137,139,140,141,142,143,144,146,147,148,150,152,157,158,159,160,161,162,163,165,166,167,168,169,170,171,172,173,174) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.concat([pd.read_csv(f, encoding='latin-1') for f in archivos_csv], ignore_index=True)


['c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\APPLE.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\corporativos.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\Datos1.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\FarmaciasGuadalajara.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\GOLF.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\Liverpool.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\PalacioDeHierro.csv', 'c:\\Users\\xboxn\\Documents\\QgisProy\\DATA\\C1_IngresosAltos\\ZARA.csv']


In [10]:
from cartopy.io import shapereader

# Descarga y cachea los archivos necesarios
shapereader.natural_earth(resolution='50m', category='cultural', name='admin_0_boundary_lines_land')
shapereader.natural_earth(resolution='50m', category='physical', name='coastline')

WindowsPath('C:/Users/xboxn/.local/share/cartopy/shapefiles/natural_earth/physical/ne_50m_coastline.shp')

In [11]:
# -------------------------------------------------------------------
# 1. PREPARACIÓN DE DATOS
# -------------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_final[["Longitud", 'Latitud']])
y = df_final["clase"]

param_grid = {
    'n_neighbors': [10,12,13,14,15,16,17,18],
    'weights': ['distance'],
    'p': [1, 2]
}

seeds = random.sample(range(1, 10000), 10)
resultados = []

mejor_score_global = -1
mejor_modelo_global = None
mejor_X_resampled = None

print("Buscando el mejor modelo...")

# -------------------------------------------------------------------
# 2. BÚSQUEDA DEL MEJOR MODELO (LOOP DE SEMILLAS)
# -------------------------------------------------------------------
for seed in seeds:
    print('.', end='', flush=True) 
    
    rus = RandomUnderSampler(random_state=seed)
    X_resampled, y_resampled = rus.fit_resample(X_scaled, y)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.2, random_state=seed, stratify=y_resampled
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    knn = KNeighborsClassifier()
    grid_search = GridSearchCV(knn, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=-1, verbose=0)
    grid_search.fit(X_train, y_train)
    
    modelo_actual = grid_search.best_estimator_
    y_pred = modelo_actual.predict(X_test)
    score_actual = balanced_accuracy_score(y_test, y_pred)
    
    resultados.append({
        "seed": seed,
        "params": grid_search.best_params_,
        "score": score_actual
    })
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_modelo_global = modelo_actual
        mejor_X_resampled = X_resampled

df_resultados = pd.DataFrame(resultados)
mejor = df_resultados.loc[df_resultados["score"].idxmax()]

print("\n\n=== Mejor modelo encontrado ===")
print(f"Seed: {mejor['seed']}")
print(f"Parámetros: {mejor['params']}")
print(f"Balanced Accuracy: {mejor['score']:.4f}")



# -------------------------------------------------------------------
# 3. CREACIÓN DE LA MALLA DIRECTAMENTE EN LAMBERT (LCC)
# -------------------------------------------------------------------
clf = mejor_modelo_global

# Cadenas de proyección
crs_wgs84 = "EPSG:4326"
proj_lcc_str = "+proj=lcc +lat_1=17.5 +lat_2=29.5 +lat_0=12 +lon_0=-102 +x_0=2500000 +y_0=0 +ellps=GRS80 +units=m +no_defs"

transformer_to_lcc = pyproj.Transformer.from_crs(crs_wgs84, proj_lcc_str, always_xy=True)
transformer_to_wgs = pyproj.Transformer.from_crs(proj_lcc_str, crs_wgs84, always_xy=True)

# límites geográficos reales de los datos
puntos_reales = scaler.inverse_transform(mejor_X_resampled)
lon_min, lon_max = puntos_reales[:, 0].min() - 0.05, puntos_reales[:, 0].max() + 0.05
lat_min, lat_max = puntos_reales[:, 1].min() - 0.05, puntos_reales[:, 1].max() + 0.05


esquinas_lon = [lon_min, lon_max, lon_min, lon_max]
esquinas_lat = [lat_min, lat_min, lat_max, lat_max]
esquinas_x_lcc, esquinas_y_lcc = transformer_to_lcc.transform(esquinas_lon, esquinas_lat)

x_lcc_min, x_lcc_max = min(esquinas_x_lcc), max(esquinas_x_lcc)
y_lcc_min, y_lcc_max = min(esquinas_y_lcc), max(esquinas_y_lcc)

resolucion_metros = 1000

# 3.4 Crear la malla matemática directamente en metros 
x_lcc_grid = np.arange(x_lcc_min, x_lcc_max, resolucion_metros)
y_lcc_grid = np.arange(y_lcc_max, y_lcc_min, -resolucion_metros)
xx_lcc, yy_lcc = np.meshgrid(x_lcc_grid, y_lcc_grid)

# -------------------------------------------------------------------
# 4. PREDICCIÓN SOBRE LA MALLA
# -------------------------------------------------------------------
print("\nGenerando predicciones para el Raster...")

# 4.1 Para predecir, necesitamos regresar temporalmente los puntos de la malla a Lat/Lon
lon_malla, lat_malla = transformer_to_wgs.transform(xx_lcc.ravel(), yy_lcc.ravel())

# 4.2 Escalar los puntos tal cual lo hicimos en el entrenamiento
df_malla = pd.DataFrame({'Longitud': lon_malla, 'Latitud': lat_malla})
malla_scaled = scaler.transform(df_malla)

# 4.3 Predecir las clases (1, 2 o 3)
Z_flat = clf.predict(malla_scaled)

# 4.4 Reformar a la matriz de la imagen 2D y asegurar que sea un entero
Z_raster = Z_flat.reshape(xx_lcc.shape).astype(np.int16)

# -------------------------------------------------------------------
# 5. EXPORTAR A QGIS COMO RASTER (GEOTIFF) NATIVO EN LCC
# -------------------------------------------------------------------
nombre_raster = os.path.join(os.getcwd() + r"\Resultados", "clases_socioeconomicas.tif")

# Definir la transformación geométrica del raster (Esquina superior izquierda y resolución)
transformacion = from_origin(x_lcc_min, y_lcc_max, resolucion_metros, resolucion_metros)

with rasterio.open(
    nombre_raster,
    'w',
    driver='GTiff',
    height=Z_raster.shape[0],
    width=Z_raster.shape[1],
    count=1,          
    dtype=Z_raster.dtype,
    crs=proj_lcc_str,     # Asignamos la proyección de Lambert
    transform=transformacion,
    nodata=0  
) as dst:
    dst.write(Z_raster, 1)

print(f"¡Exportación de Raster exitosa! Arrastra '{nombre_raster}' directamente a QGIS.")

Buscando el mejor modelo...
.

.........

=== Mejor modelo encontrado ===
Seed: 7921
Parámetros: {'n_neighbors': 18, 'p': 2, 'weights': 'distance'}
Balanced Accuracy: 0.6501

Generando predicciones para el Raster...
¡Exportación de Raster exitosa! Arrastra 'c:\Users\xboxn\Documents\QgisProy\Resultados\clases_socioeconomicas.tif' directamente a QGIS.
